# E6: transcribe a VoxPopuli sample for the data quality checks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivannamor/multilingual-asr-lab/blob/main/notebooks/e6_quality_colab.ipynb)

Experiment E6 of [RESULTS.md](https://github.com/ivannamor/multilingual-asr-lab/blob/main/RESULTS.md)
looks for problems in training data: empty or wrong transcripts, clipped or noisy audio, duplicates.
One of its features is how well a strong model agrees with each transcript, which needs a GPU.

This notebook downloads the sample of Italian VoxPopuli listed in
[`experiments/E6/quality.toml`](https://github.com/ivannamor/multilingual-asr-lab/blob/main/experiments/E6/quality.toml)
(three row groups of the training set, about 10 hours, at a pinned dataset revision) and transcribes
it with Whisper-large-v3-turbo and the decoding options selected in E2. Every other feature, the
filters and the clusters are computed locally from the saved predictions.

1. On Google Colab: *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. It takes
   about 30 minutes; keep the tab open.
2. At the end the browser downloads `e6_run.zip` (if it does not, download it from the *Files*
   panel). It contains `predictions.jsonl` and `run.json`, no audio.
3. In a local clone, from the repository root: `python3 -m zipfile -e e6_run.zip .`

If the session is interrupted, run all cells again: the transcription resumes where it stopped.


## 1. Clone and install

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Work inside a clone of the repository, so that every run.json records the git commit and the
# run directories end up where the experiment scripts expect them.
if Path("../pyproject.toml").exists():  # started from notebooks/ in a local clone
    os.chdir("..")
elif not Path("pyproject.toml").exists():  # on Colab
    if not Path("multilingual-asr-lab").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ivannamor/multilingual-asr-lab"],
            check=True,
        )
    os.chdir("multilingual-asr-lab")
if importlib.util.find_spec("asr_lab") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "."], check=True)
print(Path.cwd())

## 2. Settings

In [ ]:
SPEC_PATH = Path("experiments/E6/quality.toml")  # which part of VoxPopuli, pinned
SWEEP_PATH = Path("experiments/E2/sweep.toml")  # the model and its decoding options...
SELECTION_PATH = Path("experiments/E2/results/selection_dev.json")  # ...as selected in E2
RUN_DIR = Path("experiments/runs/e6-turbo-voxpopuli")

### GPU setup

Same setup as the quickstart notebook: install the CUDA 12 libraries that CTranslate2 is built for
and load them explicitly. Unlike the quickstart, this notebook needs the GPU and stops if it is not
usable.

In [ ]:
import ctypes

import ctranslate2
import numpy as np

if ctranslate2.get_cuda_device_count() == 0:
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> T4 GPU")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"],
    check=True,
)

In [ ]:
import nvidia.cublas
import nvidia.cudnn
from faster_whisper import WhisperModel

pending = [
    path
    for package in (nvidia.cublas, nvidia.cudnn)
    for path in sorted(Path(next(iter(package.__path__))).glob("lib/*.so*"))
]
for _ in range(5):  # the libraries depend on each other: retry the ones that failed
    failed = []
    for path in pending:
        try:
            ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL)
        except OSError:
            failed.append(path)
    pending = failed
    if not pending:
        break

check = WhisperModel("tiny", device="cuda", compute_type="float16")
list(check.transcribe(np.zeros(16_000, dtype=np.float32), language="it")[0])
print("GPU ready")

## 3. Data

The row groups of the spec, read from the Hugging Face Hub at the pinned revision (about 2.4 GB) and
written as 16-bit WAV with their manifest, exactly as `asr-lab ingest-voxpopuli` does locally.


In [ ]:
from asr_lab.data.voxpopuli import ingest_voxpopuli, load_data_spec

data = load_data_spec(SPEC_PATH)
if not data.manifest.exists():
    ingest_voxpopuli(data)
print(data.manifest)

## 4. Transcription

All utterances with one model, the E2 selection (about 20 minutes).


In [ ]:
import json

from asr_lab.inference.faster_whisper import FasterWhisperModel
from asr_lab.inference.run import transcribe_manifest
from asr_lab.sweep import load_spec

sweep = load_spec(SWEEP_PATH)
selected = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))["selected"]
options = sweep.config(selected).options
print(f"{sweep.model_size}, E2 selection {selected!r}: {options}")
model = FasterWhisperModel(sweep.model_size, device="cuda", **options)
transcribe_manifest(model, data.manifest, RUN_DIR)

## 5. Download the predictions

In [ ]:
import zipfile

run = json.loads((RUN_DIR / "run.json").read_text(encoding="utf-8"))
print(f"{run['n_utterances']} utterances, RTF {run['rtf'] or float('nan'):.3f}")
archive = Path("e6_run.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for name in ("predictions.jsonl", "run.json"):
        zipped.write(RUN_DIR / name, (RUN_DIR / name).as_posix())
print(f"{archive.resolve()}: {archive.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files

    files.download(str(archive))
except ImportError:  # not on Colab
    pass